# 12 — Caching & Performance in Streamlit

## 📓 Interactive Notebook · Module 08 · Advanced

In this notebook, you'll learn:
1. **Why caching matters** in Streamlit's rerun model
2. **`@st.cache_data`** for caching function results
3. **`@st.cache_resource`** for caching shared resources
4. **Cache invalidation** and manual clearing
5. **Performance measurement** and optimization

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Explain why caching is essential for Streamlit performance
- Choose between `cache_data` and `cache_resource` appropriately
- Implement TTL, max_entries, and persistence settings
- Measure and compare cached vs. uncached performance
- Avoid common caching mistakes

## 📋 Prerequisites

- Modules 01–07 completed
- Understanding of session state and reruns
- Basic Python functions and decorators

---

## 💡 The Problem: Reruns Are Expensive

Every user interaction causes Streamlit to rerun your entire script. Without caching, expensive operations execute on every interaction.

In [ ]:
import streamlit as st
import time

st.header("⚠️ The Rerun Problem")

# This function runs on EVERY interaction
def expensive_computation():
    """Simulate a slow operation (e.g., API call, data loading)."""
    time.sleep(2)  # 2 seconds!
    return sum(i**2 for i in range(1_000_000))

# ⚠️ Without caching: runs every time
result = expensive_computation()  # 2 seconds on EVERY interaction!
st.write(f"Result: {result:,}")
st.info("💡 Click any widget above — this computation runs again!")

---

## 💡 The Solution: `@st.cache_data`

Cache function results. On subsequent calls with the same arguments, return the cached result.

In [ ]:
import streamlit as st
import time

st.header("✅ Solution: `@st.cache_data`")

@st.cache_data
def cached_computation():
    """Cached version — only runs once per unique arguments."""
    time.sleep(2)  # 2 seconds FIRST time only
    return sum(i**2 for i in range(1_000_000))

# ✅ With caching: runs once, then instant
result = cached_computation()
st.write(f"Result: {result:,}")

# Measure performance
start = time.time()
_ = cached_computation()  # Should be instant
elapsed = time.time() - start

st.success(f"⏱️ Cached call took {elapsed*1000:.2f}ms (vs 2000ms uncached)")

---

## 🔬 Experiment 1: Arguments Affect Caching

Cache keys are based on function arguments. Different arguments = different cache entries.

In [ ]:
import streamlit as st
import time

st.header("🔬 Experiment: Arguments & Cache Keys")

@st.cache_data
def process_data(n, operation):
    """Process data based on arguments — cached per unique combination."""
    time.sleep(1)  # Simulate work
    if operation == "square":
        return [i**2 for i in range(n)]
    elif operation == "cube":
        return [i**3 for i in range(n)]
    else:
        return [i for i in range(n)]

col1, col2 = st.columns(2)
with col1:
    n = st.slider("N", 10, 100, 50)
with col2:
    op = st.selectbox("Operation", ["square", "cube", "identity"])

start = time.time()
result = process_data(n, op)
elapsed = time.time() - start

st.write(f"**{op}({n})** — {len(result)} items")
st.write(f"Time: {elapsed*1000:.2f}ms")
st.write(f"First 10: {result[:10]}")

st.info("💡 Change N or Operation — each unique combination creates a new cache entry.")

---

## 💡 `@st.cache_resource` for Shared Resources

Use `cache_resource` for objects that should be shared (not copied):
- Database connections
- ML models
- File handles

In [ ]:
import streamlit as st
import time

st.header("✅ `@st.cache_resource` for Resources")

@st.cache_resource
def load_model(model_name):
    """Load ML model — shared singleton, loaded once per model name."""
    time.sleep(1)  # Simulate model loading
    return {"name": model_name, "version": "1.0", "loaded": True}

# Load model (cached as resource)
model = load_model("classifier_v2")
st.write(f"Model: {model}")

# Verify it's the same object (singleton)
model2 = load_model("classifier_v2")
st.write(f"Same object? {model is model2}")  # True!

st.info("💡 Resources are shared across all users and sessions.")

### Key Difference: Copies vs. Singletons

| Decorator | Returns | Use For |
|-----------|---------|----------|
| `@st.cache_data` | **Copies** | Data (DataFrames, dicts, lists) |
| `@st.cache_resource` | **Same object** | Resources (connections, models) |

In [ ]:
import streamlit as st

st.header("🔬 Experiment: Copies vs. Singletons")

@st.cache_data
def get_data():
    return {"value": 0}

@st.cache_resource
def get_resource():
    return {"value": 0}

# Test cache_data (returns copies)
d1 = get_data()
d2 = get_data()
d1["value"] = 100
st.write(f"cache_data: d1={d1}, d2={d2}")
st.write(f"  Same object? {d1 is d2}")  # False

# Test cache_resource (returns same object)
r1 = get_resource()
r2 = get_resource()
r1["value"] = 100
st.write(f"cache_resource: r1={r1}, r2={r2}")
st.write(f"  Same object? {r1 is r2}")  # True

st.warning("⚠️ Modifying a cache_resource affects ALL users!")

---

## 🔬 Experiment 2: TTL (Time-to-Live)

Control how long cached values persist.

In [ ]:
import streamlit as st
import time
from datetime import datetime

st.header("🔬 Experiment: TTL (Time-to-Live)")

@st.cache_data(ttl=10)  # Expires after 10 seconds
def get_timestamp():
    return datetime.now().strftime("%H:%M:%S.%f")[:-3]

@st.cache_data  # Never expires (default)
def get_fixed_value():
    return "Always the same"

col1, col2 = st.columns(2)
with col1:
    st.write("**TTL = 10 seconds**")
    ts = get_timestamp()
    st.write(f"Cached at: {ts}")
    st.caption("Wait 10 seconds and click again — value updates!")

with col2:
    st.write("**No TTL (forever)**")
    val = get_fixed_value()
    st.write(f"Value: {val}")
    st.caption("This never changes without manual clear.")

# Manual refresh
if st.button("🔄 Clear Timestamp Cache"):
    get_timestamp.clear()
    st.rerun()

---

## 🔬 Experiment 3: Max Entries

Limit cache size to prevent memory issues.

In [ ]:
import streamlit as st
import time

st.header("🔬 Experiment: Max Entries")

@st.cache_data(max_entries=3)  # Only keep last 3 entries
def process_item(item_id):
    time.sleep(0.5)  # Simulate work
    return f"Processed item {item_id}"

item_id = st.number_input("Item ID", 1, 10, 1)

if st.button("Process"):
    result = process_item(item_id)
    st.write(result)

st.info("💡 With max_entries=3, processing 4 different items evicts the oldest.")

---

## 💡 Cache Invalidation

Clear caches manually when needed.

In [ ]:
import streamlit as st
import time

st.header("🔄 Cache Invalidation")

@st.cache_data
def load_data(source):
    time.sleep(1)  # Simulate loading
    return {"source": source, "loaded": time.time()}

# Load from different sources
source = st.selectbox("Data source", ["database", "api", "file"])
data = load_data(source)
st.write(f"Data from {data['source']}: loaded at {data['loaded']:.0f}")

st.divider()

# Clearing options
col1, col2, col3 = st.columns(3)
with col1:
    if st.button("Clear this source"):
        load_data.clear(source)
        st.rerun()
with col2:
    if st.button("Clear all sources"):
        load_data.clear()
        st.rerun()
with col3:
    if st.button("Clear ALL caches"):
        st.cache_data.clear()
        st.rerun()

---

## 🎯 Practical Example: Pandas Data Pipeline

Cache each stage of a data pipeline.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import time

st.header("🎯 Practical: Pandas Pipeline")

@st.cache_data
def generate_data(n_rows):
    """Generate sample data — cached per row count."""
    time.sleep(1)  # Simulate I/O
    np.random.seed(42)
    return pd.DataFrame({
        "category": np.random.choice(["A", "B", "C", "D"], n_rows),
        "value": np.random.randn(n_rows) * 10 + 50,
        "quantity": np.random.randint(1, 100, n_rows)
    })

@st.cache_data
def filter_data(df, category):
    """Filter by category — cached per category."""
    if category == "All":
        return df
    return df[df["category"] == category]

@st.cache_data
def compute_stats(df):
    """Compute statistics — cached per input DataFrame."""
    return df.describe()

# Pipeline
n_rows = st.slider("Rows", 100, 10000, 1000)
category = st.selectbox("Category", ["All", "A", "B", "C", "D"])

df = generate_data(n_rows)          # Cached
filtered = filter_data(df, category)  # Cached
stats = compute_stats(filtered)        # Cached

st.dataframe(filtered.head(10))
st.write("**Statistics:**")
st.dataframe(stats)

st.info("💡 Each pipeline stage is cached independently — changes to category don't regenerate data.")

---

## 🎯 Practical Example: Model Loading

Cache ML model as a shared resource.

In [ ]:
import streamlit as st
import time
import numpy as np

st.header("🎯 Practical: Model Loading")

@st.cache_resource
def load_model(model_type):
    """Load ML model — shared resource, loaded once."""
    time.sleep(2)  # Simulate model loading
    # In real code: return joblib.load(f"{model_type}.joblib")
    return {"type": model_type, "features": 10, "classes": 3}

@st.cache_data
def predict(model, features):
    """Run prediction — cached per features."""
    time.sleep(0.5)  # Simulate inference
    # In real code: return model.predict(features)
    return np.random.randint(0, model["classes"], size=len(features))

# Load model
model_type = st.selectbox("Model", ["random_forest", "svm", "neural_net"])
model = load_model(model_type)
st.write(f"Loaded: {model}")

# Predict
n_samples = st.slider("Samples", 1, 100, 10)
features = np.random.randn(n_samples, model["features"])

if st.button("Predict"):
    predictions = predict(model, features.tolist())
    st.write(f"Predictions: {predictions}")

---

## ⚠️ Common Caching Mistakes

### Mistake 1: Caching Non-Pickleable Objects

```python
# ❌ WRONG: Database connection isn't pickleable
@st.cache_data
def get_connection():
    return sqlite3.connect("data.db")

# ✅ CORRECT: Use cache_resource
@st.cache_resource
def get_connection():
    return sqlite3.connect("data.db")
```

### Mistake 2: Caching Side Effects

```python
# ❌ WRONG: Side effects won't repeat on cache hit
@st.cache_data
def log_and_return(value):
    print(f"Processing {value}")  # Only prints first time!
    return value * 2
```

### Mistake 3: Missing TTL for Real-Time Data

```python
# ⚠️ Stale data
@st.cache_data
def get_stock_price():
    return api.get_price("AAPL")  # Never updates!

# ✅ Better
@st.cache_data(ttl=60)
def get_stock_price():
    return api.get_price("AAPL")  # Updates every minute
```

---

## 🎯 Performance Measurement

Compare cached vs. uncached performance.

In [ ]:
import streamlit as st
import time

st.header("📊 Performance Comparison")

def slow_computation(n):
    """Uncached version."""
    time.sleep(0.5)
    return sum(i**2 for i in range(n))

@st.cache_data
def fast_computation(n):
    """Cached version."""
    time.sleep(0.5)
    return sum(i**2 for i in range(n))

n = st.slider("Compute up to N", 10000, 1000000, 100000, step=10000)

col1, col2 = st.columns(2)

with col1:
    st.write("**Without Cache**")
    start = time.time()
    result1 = slow_computation(n)
    t1 = time.time() - start
    st.write(f"Time: {t1*1000:.0f}ms")

with col2:
    st.write("**With Cache**")
    start = time.time()
    result2 = fast_computation(n)
    t2 = time.time() - start
    st.write(f"Time: {t2*1000:.2f}ms")

st.metric("Speedup", f"{t1/t2:.0f}x" if t2 > 0 else "∞")

---

## 🎯 Challenges

### Challenge 1: Cached API Wrapper
Create a function that fetches data from an API and caches it with:
- 5-minute TTL
- Max 10 entries
- Custom spinner text

### Challenge 2: Database Query Cache
Build a query executor that:
- Caches results per query string
- Provides manual refresh button
- Shows cache hit/miss statistics

### Challenge 3: Model Registry
Create a model loader that:
- Caches multiple models by name
- Allows clearing individual models
- Shows which models are loaded

In [ ]:
# Challenge 1: Cached API Wrapper
import streamlit as st
import time

# TODO: Create a function with @st.cache_data that:
# - Takes a URL parameter
# - TTL of 5 minutes
# - Max 10 entries
# - Custom spinner "Fetching data..."
# Simulate API call with time.sleep(1)


---

## 📝 Key Takeaways

1. **Caching is essential** — without it, expensive operations run on every interaction.

2. **`@st.cache_data`** — returns **copies**, use for data (DataFrames, queries).

3. **`@st.cache_resource`** — returns **same object**, use for resources (connections, models).

4. **TTL controls freshness** — set appropriate expiration for time-sensitive data.

5. **Manual clearing** — `func.clear()` and `st.cache_data.clear()` for on-demand refresh.

6. **Performance measurement** — use `show_time=True` or manual timing to verify effectiveness.

### Quick Reference

| Feature | `cache_data` | `cache_resource` |
|---------|--------------|------------------|
| Returns | Copies | Same object |
| Use for | Data | Resources |
| Pickleable? | Required | Not required |
| Thread-safe? | Not required | Required (global) |

---

## 📚 Further Reading

- [Streamlit Caching Overview](https://docs.streamlit.io/develop/concepts/architecture/caching)
- [st.cache_data](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_data)
- [st.cache_resource](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource)

---

## 🔗 Related Materials

- 📖 Reading: [12 — Caching & Performance](../readings/12_caching_and_performance.md)
- ✏️ Exercise: [12 — Caching Workshop](../exercises/12_caching_workshop.py)
- 🖥️ Demo App: [12 — Caching Demo](../apps/12_caching_performance_demo.py)
- 📝 Quiz: [08 — Caching & Performance](../quizzes/08_caching_performance.md)